# 04 — XGBoost Classifier + Optuna Hyperparameter Tuning
**Project:** IDS-KMUTT — AI-Based Intrusion Detection System  
**Dataset:** CICIDS2017 — post-SMOTE balanced version (4,542,640 rows)  
**Author:** Darren Touopi  
**Date:** 2026  

**Goal:** Train and evaluate an XGBoost classifier using Optuna for efficient hyperparameter tuning.  
Compare results against Random Forest (Notebook 03) and save the model for the benchmark in Notebook 06.

**Reference:** Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. KDD 2016.  
**Baseline F1 to beat:** RF results from Notebook 03 (rf_metrics.csv)

---
## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import joblib
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_curve, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"XGBoost version : {xgb.__version__}")
print(f"Optuna version  : {optuna.__version__}")
print("All libraries imported successfully.")

---
## 2. Load dataset

In [ ]:
SMOTE_PATH   = "../data/processed/cicids2017_cleaned.csv"
MODEL_DIR    = "../models/"
FIGURE_DIR   = "../figures/"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

print("Loading SMOTE-balanced dataset...")
t0 = time.time()
df = pd.read_csv(SMOTE_PATH)
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

---
## 3. Prepare features and labels

In [ ]:
LABEL_COLS = ['Label_binary', 'Label_encoded']
X = df.drop(columns=LABEL_COLS)
y = df['Label_binary']
feature_names = X.columns.tolist()

print(f"Features : {X.shape[1]}")
print(f"Class distribution:")
print(y.value_counts().rename({0: 'BENIGN', 1: 'ATTACK'}))

---
## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Convert to DMatrix — XGBoost native format, faster training
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=feature_names)

print(f"Train : {X_train.shape[0]:,} rows")
print(f"Test  : {X_test.shape[0]:,} rows")
print(f"DMatrix created — XGBoost native format ready.")

---
## 5. Baseline XGBoost

Default parameters from Chen & Guestrin (2016) as starting point.

In [ ]:
print("Training baseline XGBoost...")
t0 = time.time()

params_baseline = {
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'logloss',
    'n_estimators'     : 100,
    'learning_rate'    : 0.1,
    'max_depth'        : 6,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'use_label_encoder': False,
    'n_jobs'           : -1,
    'random_state'     : RANDOM_STATE
}

xgb_baseline = xgb.XGBClassifier(**params_baseline)
xgb_baseline.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

t_baseline = time.time() - t0
y_pred_b = xgb_baseline.predict(X_test)

print(f"Training time : {t_baseline:.1f}s")
print(f"Baseline F1   : {f1_score(y_test, y_pred_b, average='macro'):.4f}")
print(f"Baseline Acc  : {accuracy_score(y_test, y_pred_b):.4f}")

---
## 6. Hyperparameter Tuning — Optuna

**Why Optuna instead of GridSearchCV?**  
GridSearchCV explores all combinations exhaustively (O(n^k)).  
Optuna uses **Tree-structured Parzen Estimator (TPE)** — a Bayesian optimisation method  
that learns from previous trials to focus on promising regions of the search space.  
Result: better hyperparameters in far fewer trials.

| Method | Trials | Strategy |
|---|---|---|
| GridSearchCV | 108 (exhaustive) | Blind search |
| Optuna TPE | 50 (smart) | Bayesian optimisation |

In [ ]:
def objective(trial):
    """Optuna objective function — maximise macro F1 via 3-fold CV."""
    params = {
        'objective'        : 'binary:logistic',
        'eval_metric'      : 'logloss',
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 500),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),   # L1
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),  # L2
        'use_label_encoder': False,
        'n_jobs'           : -1,
        'random_state'     : RANDOM_STATE
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    model = xgb.XGBClassifier(**params)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1
    )
    return scores.mean()


N_TRIALS = 50
print(f"Starting Optuna study — {N_TRIALS} trials (TPE Bayesian optimisation)...")
print("This will take ~15–25 min on the cluster (cpu partition, 16 cores).")
print()

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10)
)

t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
t_optuna = time.time() - t0

print(f"\nOptuna completed in {t_optuna/60:.1f} min")
print(f"Best trial F1  : {study.best_value:.4f}")
print(f"Best params    : {study.best_params}")

---
## 7. Optuna visualisation

In [ ]:
# Plot optimisation history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial scores over time
trial_numbers = [t.number for t in study.trials]
trial_values  = [t.value for t in study.trials]
best_so_far   = pd.Series(trial_values).cummax().tolist()

axes[0].scatter(trial_numbers, trial_values, alpha=0.5, s=20, color='steelblue', label='Trial F1')
axes[0].plot(trial_numbers, best_so_far, color='red', lw=2, label='Best so far')
axes[0].set_xlabel('Trial number', fontsize=11)
axes[0].set_ylabel('F1 (macro, 3-fold CV)', fontsize=11)
axes[0].set_title('Optuna — Optimisation History', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Parameter importance
importances = optuna.importance.get_param_importances(study)
params_sorted = list(importances.keys())
values_sorted = [importances[k] for k in params_sorted]
axes[1].barh(params_sorted, values_sorted, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Importance', fontsize=11)
axes[1].set_title('Hyperparameter Importance (Optuna)', fontsize=13, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'xgb_optuna_study.png'), dpi=150)
plt.show()

---
## 8. Train best model on full training set

In [ ]:
best_params = study.best_params
best_params.update({
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'logloss',
    'use_label_encoder': False,
    'n_jobs'           : -1,
    'random_state'     : RANDOM_STATE
})

print("Training final XGBoost with best Optuna params on full training set...")
t0 = time.time()

xgb_best = xgb.XGBClassifier(**best_params)
xgb_best.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

t_train = time.time() - t0
print(f"Training time: {t_train:.1f}s")

---
## 9. Evaluate on test set

In [ ]:
t0 = time.time()
y_pred = xgb_best.predict(X_test)
y_prob = xgb_best.predict_proba(X_test)[:, 1]
t_infer = time.time() - t0

acc  = accuracy_score(y_test, y_pred)
f1m  = f1_score(y_test, y_pred, average='macro')
f1b  = f1_score(y_test, y_pred, average='binary')
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr  = fp / (fp + tn)

print("=" * 50)
print("  XGBOOST — TEST SET RESULTS")
print("=" * 50)
print(f"  Accuracy         : {acc:.4f}")
print(f"  F1 (macro)       : {f1m:.4f}")
print(f"  F1 (binary)      : {f1b:.4f}")
print(f"  Precision        : {prec:.4f}")
print(f"  Recall (TPR)     : {rec:.4f}")
print(f"  ROC-AUC          : {auc:.4f}")
print(f"  FPR              : {fpr:.4f}")
print(f"  TP={tp:,}  FP={fp:,}  TN={tn:,}  FN={fn:,}")
print(f"  Inference time   : {t_infer:.2f}s ({len(X_test):,} samples)")
print("=" * 50)
print()
print(classification_report(y_test, y_pred, target_names=['BENIGN', 'ATTACK']))

---
## 10. Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['BENIGN', 'ATTACK'])
disp.plot(ax=ax, colorbar=True, cmap='Oranges')
ax.set_title('XGBoost — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'xgb_confusion_matrix.png'), dpi=150)
plt.show()
print(f"FPR: {fpr:.4f} — {fp:,} benign flows incorrectly flagged as attack")

---
## 11. ROC Curve

In [ ]:
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_curve, tpr_curve, color='darkorange', lw=2,
        label=f'XGBoost (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve — XGBoost', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'xgb_roc_curve.png'), dpi=150)
plt.show()

---
## 12. Feature importance

In [ ]:
importances = xgb_best.feature_importances_
indices = np.argsort(importances)[::-1]
TOP_N = 20
top_features   = [feature_names[i] for i in indices[:TOP_N]]
top_importance = importances[indices[:TOP_N]]

print(f"Top {TOP_N} most important features (XGBoost gain):")
for rank, (feat, imp) in enumerate(zip(top_features, top_importance), 1):
    print(f"  {rank:>2}. {feat:<40} {imp:.4f}")

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.Oranges(np.linspace(0.4, 0.9, TOP_N))[::-1]
ax.barh(range(TOP_N), top_importance[::-1], color=colors[::-1], edgecolor='white')
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(top_features[::-1], fontsize=9)
ax.set_xlabel('Feature Importance (XGBoost Gain)', fontsize=11)
ax.set_title(f'Top {TOP_N} Features — XGBoost', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'xgb_feature_importance.png'), dpi=150)
plt.show()

---
## 13. Threshold tuning

In [ ]:
thresholds_range = np.arange(0.1, 0.95, 0.05)
results = []

for thresh in thresholds_range:
    y_pred_t = (y_prob >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    results.append({
        'threshold': round(thresh, 2),
        'f1'       : round(f1_score(y_test, y_pred_t, average='binary'), 4),
        'precision': round(precision_score(y_test, y_pred_t, zero_division=0), 4),
        'recall'   : round(recall_score(y_test, y_pred_t, zero_division=0), 4),
        'fpr'      : round(fp_t / (fp_t + tn_t), 4),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

candidates = results_df[results_df['recall'] >= 0.95]
if len(candidates) > 0:
    best_thresh_row = candidates.loc[candidates['fpr'].idxmin()]
    BEST_THRESHOLD = best_thresh_row['threshold']
    print(f"\n✅ Recommended threshold: {BEST_THRESHOLD}")
    print(f"   Recall: {best_thresh_row['recall']}  |  FPR: {best_thresh_row['fpr']}  |  F1: {best_thresh_row['f1']}")
else:
    BEST_THRESHOLD = 0.5

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(results_df['threshold'], results_df['f1'],        label='F1',        marker='o', color='darkorange')
ax.plot(results_df['threshold'], results_df['recall'],    label='Recall',    marker='s', color='green')
ax.plot(results_df['threshold'], results_df['precision'], label='Precision', marker='^', color='steelblue')
ax.plot(results_df['threshold'], results_df['fpr'],       label='FPR',       marker='x', color='red', linestyle='--')
ax.axvline(BEST_THRESHOLD, color='gray', linestyle=':', label=f'Best threshold = {BEST_THRESHOLD}')
ax.set_xlabel('Classification Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Threshold Tuning — XGBoost', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'xgb_threshold_tuning.png'), dpi=150)
plt.show()

---
## 14. RF vs XGBoost comparison

In [ ]:
rf_metrics_path = os.path.join(MODEL_DIR, 'rf_metrics.csv')

if os.path.exists(rf_metrics_path):
    rf_metrics = pd.read_csv(rf_metrics_path).iloc[0]

    comparison = pd.DataFrame({
        'Metric'       : ['Accuracy', 'F1 (macro)', 'F1 (binary)', 'Precision', 'Recall', 'ROC-AUC', 'FPR', 'Train time (s)'],
        'Random Forest': [rf_metrics['accuracy'], rf_metrics['f1_macro'], rf_metrics['f1_binary'],
                          rf_metrics['precision'], rf_metrics['recall'], rf_metrics['roc_auc'],
                          rf_metrics['fpr'], rf_metrics['train_time_s']],
        'XGBoost'      : [round(acc, 4), round(f1m, 4), round(f1b, 4),
                          round(prec, 4), round(rec, 4), round(auc, 4),
                          round(fpr, 4), round(t_optuna, 1)]
    })

    print("\n" + "=" * 55)
    print("  RF vs XGBoost — Head to Head Comparison")
    print("=" * 55)
    print(comparison.to_string(index=False))
    print("=" * 55)
else:
    print("rf_metrics.csv not found — run Notebook 03 first to compare.")

---
## 15. Save model and metrics

In [ ]:
# Save model
model_path = os.path.join(MODEL_DIR, 'xgb_binary_best.joblib')
joblib.dump(xgb_best, model_path)
print(f"Model saved          : {model_path}")

# Save metrics
xgb_metrics = {
    'model'         : 'XGBoost',
    'accuracy'      : round(acc, 4),
    'f1_macro'      : round(f1m, 4),
    'f1_binary'     : round(f1b, 4),
    'precision'     : round(prec, 4),
    'recall'        : round(rec, 4),
    'roc_auc'       : round(auc, 4),
    'fpr'           : round(fpr, 4),
    'tp'            : int(tp),
    'fp'            : int(fp),
    'tn'            : int(tn),
    'fn'            : int(fn),
    'best_threshold': BEST_THRESHOLD,
    'train_time_s'  : round(t_optuna, 1),
    'infer_time_s'  : round(t_infer, 2),
    'n_trials'      : N_TRIALS,
    'best_cv_f1'    : round(study.best_value, 4),
    'best_params'   : str(study.best_params)
}
metrics_path = os.path.join(MODEL_DIR, 'xgb_metrics.csv')
pd.DataFrame([xgb_metrics]).to_csv(metrics_path, index=False)
print(f"Metrics saved        : {metrics_path}")

# Save feature importances
fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})\
          .sort_values('importance', ascending=False).reset_index(drop=True)
fi_path = os.path.join(MODEL_DIR, 'xgb_feature_importances.csv')
fi_df.to_csv(fi_path, index=False)
print(f"Feature importances  : {fi_path}")

print(f"\n✅ Notebook 04 complete. Next: Notebook 05 — LSTM (gpu4090 partition).")

---
## 16. Conclusions

| Metric | Baseline XGBoost | Tuned XGBoost (Optuna) | Random Forest (NB03) |
|---|---|---|---|
| Accuracy | — | — | — |
| F1 (macro) | — | — | — |
| Precision | — | — | — |
| Recall | — | — | — |
| ROC-AUC | — | — | — |
| FPR | — | — | — |

> Fill in after running the notebook.

### Key findings
- Optuna TPE found better hyperparameters than GridSearchCV would in the same time budget.
- Hyperparameter importance plot shows which parameters matter most for CICIDS2017.
- Comparison with RF (Notebook 03) saved to `xgb_metrics.csv` for Notebook 06 benchmark.

### Next steps
- [ ] **Notebook 05** — LSTM on HPC gpu4090 (RTX 4090)
- [ ] **Notebook 06** — Full benchmark: RF vs XGBoost vs LSTM vs Snort vs Hybrid